# ⚡ Tự Động Hóa Backtest Siêu Tốc Trên Google Colab Pro (High-RAM)
### Phiên Bản Tối Ưu: **Smart Caching Layer** (Setup < 30 Giây | Backtest ~3-4 Phút)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tminhnhat/NostalgiaForInfinity/blob/main/notebooks/Run_NFIX8_Backtest_Colab_HighRAM.ipynb)

---
### 📁 CẤU TRÚC THƯ MỤC CACHE TRÊN GOOGLE DRIVE (`botx8_cache/`):
1. **`binance_candles_185d.tar.gz`** (Dữ liệu nến 185 ngày): Kéo thả từ máy tính cá nhân lên Drive 1 lần duy nhất (hoặc để Colab tự tải 1 lần).
2. **`talib_c.tar.gz` & `wheels_cache.tar.gz`** (Thư viện Linux x86_64): **Colab tự động biên dịch và tự lưu lên Drive** ở lần chạy đầu tiên. *(Không copy từ Windows vì không tương thích)*.

---
### 🚀 BẢNG SO SÁNH TỐC ĐỘ TRƯỚC VÀ SAU KHI TỐI ƯU:
| Hạng mục | Quy trình cũ (chậm) | Quy trình tối ưu với Smart Cache | Cải thiện |
| :--- | :--- | :--- | :--- |
| **Cài TA-Lib C & Pip** | ~3 - 5 phút (compile C & tải 50 gói) | **~5 - 8 giây** (nạp từ Drive cache) | **Nhanh gấp 30 lần** |
| **Nạp dữ liệu nến 185 ngày** | ~10 - 20 phút (tải 18,500 file zip) | **~3 giây** (giải nén bundle nến) | **Nhanh gấp 300 lần** ⚡ |
| **Chạy Backtest 185 ngày** | ~3 - 4 phút (8 vCPUs) | **~3 - 4 phút** (8 vCPUs) | Tối đa công suất |
| **Tổng thời gian từ A - Z** | **15 - 25 phút** | **~3.5 - 4.5 phút** | **Tiết kiệm 85% thời gian** |

> 💡 **CÁCH SỬ DỤNG 1-CLICK:**
> 1. Vào **Runtime ➔ Change runtime type** ➔ Chọn **None (CPU)** và **High-RAM**.
> 2. Bấm **Ctrl + F9** (hoặc chọn menu **Runtime ➔ Run all**).
> 3. Tận hưởng: Toàn bộ quá trình chạy tự động ➔ Gửi báo cáo về Telegram ➔ Tự ngắt kết nối Colab để tiết kiệm điểm!

In [ ]:
# ==============================================================================
# BƯỚC 1: KIỂM TRA PHẦN CỨNG & KẾT NỐI GOOGLE DRIVE ĐỂ DÙNG CACHE
# ==============================================================================
import os
import psutil
from pathlib import Path

vcpus = os.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"🖥️  Số luồng CPU (vCPUs): {vcpus} cores")
print(f"🧠  Dung lượng RAM:       {ram_gb:.1f} GB")

# Kết nối Google Drive để lưu và đọc cache siêu tốc
CACHE_DIR = Path('/content/drive/MyDrive/botx8_cache')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_ACTIVE = True
    print(f"✅ Đã kết nối Google Drive! Thư mục cache: {CACHE_DIR}")
except Exception as e:
    DRIVE_ACTIVE = False
    print(f"⚠️ Không mount được Google Drive ({e}). Sẽ sử dụng local cache tạm thời.")

In [ ]:
# ==============================================================================
# BƯỚC 2: CÀI ĐẶT TA-LIB & THƯ VIỆN BẰNG CACHE (GIẢM TỪ 4 PHÚT XUỐNG 8 GIÂY)
# ==============================================================================
import os
import sys
import subprocess
import time
from pathlib import Path

TALIB_C_CACHE = CACHE_DIR / 'talib_c.tar.gz' if DRIVE_ACTIVE else Path('/content/talib_c.tar.gz')
WHEELS_CACHE = CACHE_DIR / 'wheels_cache.tar.gz' if DRIVE_ACTIVE else Path('/content/wheels_cache.tar.gz')
PIP_WHEELS_DIR = Path('/root/.cache/pip/wheels')
PIP_WHEELS_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
has_cache = TALIB_C_CACHE.exists() and WHEELS_CACHE.exists()

if has_cache:
    print('⚡ TÌM THẤY CACHE THƯ VIỆN LINUX TRÊN GOOGLE DRIVE!')
    print('⏳ Đang nạp TA-Lib C và giải nén Python Wheels...')
    !tar -xzf {TALIB_C_CACHE} -C / 2>/dev/null
    !ldconfig 2>/dev/null
    !tar -xzf {WHEELS_CACHE} -C /root/.cache/pip/ 2>/dev/null
    !pip install -q --find-links=/root/.cache/pip/wheels freqtrade ta-lib lightgbm scikit-learn joblib python-rapidjson pyarrow fastparquet
    print('✅ Đã nạp môi trường từ cache!')

# Tự động kiểm tra an toàn: Nếu TA-Lib C chưa có (cache hỏng), tự động biên dịch lại ngay
if not os.path.exists('/usr/lib/libta_lib.so.0'):
    print('⏳ Đang biên dịch TA-Lib C library...')
    install_talib_script = '''
    wget -q http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
    tar -xzf ta-lib-0.4.0-src.tar.gz
    cd ta-lib/
    ./configure --prefix=/usr > /dev/null 2>&1
    make -j$(nproc) > /dev/null 2>&1
    sudo make install > /dev/null 2>&1
    sudo ldconfig > /dev/null 2>&1
    cd ..
    rm -rf ta-lib ta-lib-0.4.0-src.tar.gz
    '''
    subprocess.run(install_talib_script, shell=True, check=True, executable='/bin/bash')
    !pip install -q ta-lib freqtrade lightgbm scikit-learn joblib python-rapidjson pyarrow fastparquet

# Kiểm tra an toàn: Đảm bảo import thành công các thư viện cốt lõi
try:
    import talib
    import freqtrade
    import lightgbm
except ImportError:
    print('⚠️ Phát hiện thiếu package, đang tự động cài bổ sung từ PyPI...')
    !pip install -q ta-lib freqtrade lightgbm scikit-learn joblib python-rapidjson pyarrow fastparquet

# Tự động cập nhật cache mới chuẩn lên Google Drive nếu chưa có hoặc vừa tạo mới
if DRIVE_ACTIVE and not has_cache:
    print('💾 Đang tự động lưu cache thư viện Linux hoàn chỉnh lên Google Drive...')
    try:
        !tar -czf {TALIB_C_CACHE} /usr/lib/libta_lib* /usr/include/ta-lib 2>/dev/null
        !tar -czf {WHEELS_CACHE} -C /root/.cache/pip wheels 2>/dev/null
        print(f'✅ Đã lưu cache thư viện vào: {CACHE_DIR}')
    except Exception as e:
        print(f'⚠️ Lưu cache thư viện: {e}')

# Xác minh cuối cùng
import talib
import freqtrade
import lightgbm
print(f'\n🎉 MÔI TRƯỜNG HOÀN TOÀN SẴN SÀNG TRONG {time.time()-t0:.1f}s!')
print(f'   - Freqtrade: {freqtrade.__version__}')
print(f'   - TA-Lib:    {talib.__version__}')
print(f'   - LightGBM:  {lightgbm.__version__}')


In [ ]:
# ==============================================================================
# BƯỚC 3: CLONE HOẶC CẬP NHẬT MÃ NGUỒN BOT MỚI NHẤT TỪ GITHUB
# ==============================================================================
import os
from pathlib import Path

REPO_URL = "https://github.com/tminhnhat/NostalgiaForInfinity.git"
REPO_DIR = "/content/NostalgiaForInfinity"

if not os.path.exists(REPO_DIR):
    print(f"📥 Đang clone repository từ: {REPO_URL}...")
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"🔄 Đang cập nhật repository mới nhất...")
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}

# Xác nhận các file chiến lược và mô hình
assert Path("NostalgiaForInfinityX8_Combined_AI.py").exists(), "Thiếu file chiến lược!"
assert Path("configs/backtest_config.json").exists(), "Thiếu file config backtest!"
assert Path("user_data/models/nfix8_short_ai_v3.pkl").exists(), "Thiếu file mô hình AI!"
print("✅ Chiến lược & Mô hình AI Gatekeeper v3 đã sẵn sàng!")

In [ ]:
# ==============================================================================
# BƯỚC 4: NẠP DỮ LIỆU NẾN SIÊU TỐC TỪ BUNDLE (GIẢM TỪ 20 PHÚT XUỐNG 3 GIÂY)
# ==============================================================================
import os
import time
from pathlib import Path

CANDLE_BUNDLE_DRIVE = CACHE_DIR / 'binance_candles_185d.tar.gz'
CANDLE_BUNDLE_LOCAL = Path('/content/binance_candles_185d.tar.gz')
DEST_DIR = Path('/content/NostalgiaForInfinity/user_data/data/binance')
DEST_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
bundle_found = None
if CANDLE_BUNDLE_DRIVE.exists():
    bundle_found = CANDLE_BUNDLE_DRIVE
elif CANDLE_BUNDLE_LOCAL.exists():
    bundle_found = CANDLE_BUNDLE_LOCAL

if bundle_found:
    print(f"⚡ TÌM THẤY GÓI DỮ LIỆU NẾN: {bundle_found}")
    print("⏳ Đang giải nén trực tiếp vào user_data/data/binance/...")
    !tar -xzf {bundle_found} -C {DEST_DIR}
    print(f"🎉 Đã nạp toàn bộ 185 ngày nến thành công chỉ trong {time.time()-t0:.1f}s!")
else:
    print("ℹ️ Chưa có gói nén nến trên Google Drive. Tiến hành tải trực tiếp từ Binance Vision...")
    !python scripts/download_candles.py \
        --config configs/backtest_config.json \
        --days 185 \
        --output-dir user_data/data/binance \
        --workers 24 \
        --skip-existing
    
    # Tự động nén và lưu gói nến vào Google Drive cho các lần sau
    if DRIVE_ACTIVE:
        print("💾 Đang tự động lưu gói nến vào Google Drive để lần sau chạy trong 3 giây...")
        !tar -czf {CANDLE_BUNDLE_DRIVE} -C user_data/data/binance futures 2>/dev/null
        print(f"✅ Đã lưu gói nến tại: {CANDLE_BUNDLE_DRIVE}")

# Áp dụng bản vá Offline Markets để tránh bị sàn chặn IP
!python scripts/patch_offline_markets.py
feather_count = len(list(Path('user_data/data/binance/futures').glob('*.feather')))
print(f"📊 Tổng cộng: {feather_count} file nến đã sẵn sàng trong thư mục futures!")

In [ ]:
# ==============================================================================
# BƯỚC 5: CHẠY BACKTEST ĐÁNH GIÁ SIÊU TỐC TRÊN 8 vCPUs (~3 - 4 PHÚT)
# ==============================================================================
import time

start_time = time.time()
print("🚀 BẮT ĐẦU CHẠY BACKTEST TRÊN 8 vCPUs (HIGH-RAM)...")

!python scripts/run_eval_backtest.py \
    --strategy NostalgiaForInfinityX8_Combined_AI \
    --days 185 \
    --run-label "Colab_Pro_SmartCache_Run"

elapsed = time.time() - start_time
print(f"\n⏱️ TỔNG THỜI GIAN THỰC THI: {elapsed / 60:.2f} phút ({elapsed:.1f} giây)!")

In [ ]:
# ==============================================================================
# BƯỚC 6: HIỂN THỊ BÁO CÁO HIỆU SUẤT TRỰC QUAN (MARKDOWN FORMATTED)
# ==============================================================================
from IPython.display import display, Markdown
from pathlib import Path

report_file = Path("reports/latest_report.md")
if report_file.exists():
    with open(report_file, "r", encoding="utf-8") as f:
        content = f.read()
    display(Markdown(content))
else:
    print("⚠️ Không tìm thấy file reports/latest_report.md!")

In [ ]:
# ==============================================================================
# BƯỚC 7: LƯU BÁO CÁO VÀO GOOGLE DRIVE & TỰ ĐỘNG GỬI QUA TELEGRAM
# ==============================================================================
import shutil
from pathlib import Path

# 1. Lưu vĩnh viễn báo cáo và file kết quả vào Google Drive
if DRIVE_ACTIVE:
    reports_drive = CACHE_DIR / 'reports'
    reports_drive.mkdir(parents=True, exist_ok=True)
    !cp -r reports/* {reports_drive}/ 2>/dev/null
    !cp -r user_data/backtest_results/* {reports_drive}/ 2>/dev/null
    print(f'💾 Đã lưu vĩnh viễn báo cáo vào Google Drive: {reports_drive}')

# 2. Bắn thông báo tóm tắt và đính kèm file latest_report.md về Telegram
!python scripts/notify_telegram.py \
    --report reports/latest_report.md \
    --context-title '⚡ BÁO CÁO BACKTEST COLAB PRO (HIGH-RAM)'


In [ ]:
# ==============================================================================
# BƯỚC 8: TỰ ĐỘNG NGẮT KẾT NỐI RUNTIME ĐỂ TIẾT KIỆM TÍN DỤNG COMPUTE UNITS
# ==============================================================================
AUTO_DISCONNECT = True

if AUTO_DISCONNECT:
    print("🛑 Đang hoàn tất và tự động ngắt kết nối Google Colab Pro...")
    try:
        from google.colab import runtime
        runtime.unassign()
        print("✅ Đã ngắt kết nối runtime an toàn!")
    except Exception as e:
        print(f"⚠️ Không thể ngắt runtime tự động: {e}")
else:
    print("ℹ️ AUTO_DISCONNECT đang tắt. Runtime vẫn giữ kết nối để anh tiếp tục làm việc.")